# Hybrid Quantum-Classical CVRP Solver
## Yale Quantum Courier Challenge 2026

This notebook is the **experimental frontend** for the hybrid solver.
All core solver logic lives in the shared modules under `instances/`.

| Module | Purpose |
|--------|----------|
| `common.py` | Distance matrix, route cost, validation |
| `decomposition.py` | Sweep, Clarke-Wright, k-means seed portfolio |
| `local_search.py` | 2-opt, 3-opt |
| `repair.py` | Inter-route relocate and swap |
| `quantum_improve.py` | Anchored local QAOA / brute-force improver |
| `hybrid_solver.py` | Full pipeline orchestration |

### Hybrid Pipeline
```
Stage A  ->  Seed portfolio (sweep x4, Clarke-Wright, k-means x3)
Stage B  ->  2-opt cleanup on every candidate route
Stage C  ->  Inter-route relocate + swap repair
Stage D  ->  Anchored local quantum improvement on bounded neighborhoods
```

**Quantum note**: All local neighborhood improvement is performed using native 
QAOA. QAOA utilizes the specified qubit budget (max_local_qaoa_qubits) to determine k.
requiring qubit budget >= 49. Set `max_local_qaoa_qubits=49` to force QAOA.

In [ ]:
import sys, os
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Add shared modules to path
INSTANCES_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'instances')
if INSTANCES_DIR not in sys.path:
    sys.path.insert(0, INSTANCES_DIR)

import numpy as np
import matplotlib.pyplot as plt

from common import (
    challenge_instances, build_distance_matrix,
    total_solution_distance, format_routes_text, validate_solution
)
from decomposition import generate_seed_portfolio, best_valid_candidate
from local_search import cleanup_solution
from repair import repair_routes
from quantum_improve import improve_all_routes_qaoa
from hybrid_solver import HybridSolver

print('Shared modules loaded successfully.')

## Challenge Instances

In [ ]:
instances = challenge_instances()
for inst in instances:
    print(f"{inst['instance_id']}: {inst['num_customers']} customers, "
          f"{inst['num_vehicles']} vehicles, capacity {inst['capacity']}")

## Stage A: Seed Portfolio

Generate multiple classical decompositions for a single instance.
Each produces a complete valid route set — we keep the best one.

In [ ]:
selected_id = 4  # Change to 1, 2, 3, or 4
inst = instances[selected_id - 1]
nodes = inst['nodes']
demands = inst['demands']
capacity = inst['capacity']
num_vehicles = inst['num_vehicles']
D = build_distance_matrix(nodes)

candidates = generate_seed_portfolio(
    nodes, demands, capacity, num_vehicles,
    sweep_offsets=4, kmeans_seeds=[42, 7, 123]
)
valid = [c for c in candidates if c['valid']]
print(f'Generated {len(candidates)} seeds, {len(valid)} valid')
for c in sorted(candidates, key=lambda x: x['total_dist']):
    flag = 'OK' if c['valid'] else 'INVALID'
    print(f"  [{flag}] {c['method']:20s} dist={c['total_dist']:.4f}")

## Stage B: 2-opt Cleanup

In [ ]:
cleaned = []
for c in candidates:
    routes = cleanup_solution(c['routes'], D)
    d = total_solution_distance(routes, D)
    v = validate_solution(routes, demands, capacity,
                          inst['num_customers'], num_vehicles)
    cleaned.append({**c, 'routes': routes, 'total_dist': round(d, 6), 'valid': v})

best = best_valid_candidate(cleaned)
print(f"Best after 2-opt: {best['method']} -> dist={best['total_dist']:.4f}")

## Stage C: Inter-route Repair

Relocate and swap customers between routes. Only improving moves accepted.

In [ ]:
import copy
routes = copy.deepcopy(best['routes'])
before_repair = total_solution_distance(routes, D)

routes, gain = repair_routes(routes, D, demands, capacity, nodes)
routes = cleanup_solution(routes, D)  # post-repair 2-opt

after_repair = total_solution_distance(routes, D)
print(f'Repair gain: {gain:.4f}')
print(f'Distance: {before_repair:.4f} -> {after_repair:.4f}')
for i, r in enumerate(routes):
    print(f"  Route {i+1}: {[0]+r+[0]}  load={sum(demands[c] for c in r)}/{capacity}")

## Stage D: Quantum Local Improvement

The quantum subroutine is a **local improver** on small anchored subroute
neighborhoods -- not a full-instance solver.

**Protocol per route:**
1. Select k customers (k = floor(sqrt(qubit_budget)))
2. Fix left and right anchor nodes
3. Build anchored position-QUBO on the k customers.
4. Optimize using QAOA circuit simulation.
5. Splice back only if route distance improves

| qubit_budget | k | method |
|---|---|---|
| 9 | 3 | QAOA |
| 25 | 5 | QAOA |
| 49 | 7 | QAOA |

In [ ]:
# By default we set MAX_QUBITS = 25 (k=5) representing 25 qubits used per neighborhood.
# Note: Full simulator execution can be slow for QAOA on large instances.
MAX_QUBITS = 25

routes_q = copy.deepcopy(routes)
dist_before_q = total_solution_distance(routes_q, D)

improved, total_gain, meta = improve_all_routes_qaoa(
    routes_q, D,
    max_local_qaoa_qubits=MAX_QUBITS,
    strategy='worst_edges',
)
improved = cleanup_solution(improved, D)  # final 2-opt

dist_after_q = total_solution_distance(improved, D)
print(f'Quantum-local gain: {total_gain:.4f}')
print(f'Distance: {dist_before_q:.4f} -> {dist_after_q:.4f}')
for i, m in enumerate(meta):
    print(f"  Route {i+1}: method={m.get('method','?')}, "
          f"k={m.get('neighborhood_size','?')}, outcome={m.get('outcome','?')}")

## Full Pipeline via HybridSolver (all 4 challenge instances)

In [ ]:
results = {}
for inst in instances:
    solver = HybridSolver(inst, max_local_qaoa_qubits=25, verbose=True)
    r = solver.solve(run_quantum=True)
    results[inst['instance_id']] = r
    s = r['stages']
    print(f"{inst['instance_id']:12s} dist={r['total_dist_final']:.4f} "
          f"valid={r['valid_final']} "
          f"cleanup_gain={s['dist_after_seed']-s['dist_after_cleanup']:.4f} "
          f"repair={s['repair_gain']:.4f} qaoa={s['qaoa_gain']:.4f}")
    print()

## Route Visualization

In [ ]:
def plot_routes(inst, routes, title=''):
    nodes = inst['nodes']
    depot = nodes[0]
    cust_xy = nodes[1:]
    D_plot = build_distance_matrix(nodes)
    total_d = total_solution_distance(routes, D_plot)

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(*depot, c='black', marker='X', s=200, zorder=5, label='Depot')
    ax.text(depot[0]+0.3, depot[1]+0.3, '0', fontsize=10)

    cx = [p[0] for p in cust_xy]
    cy = [p[1] for p in cust_xy]
    ax.scatter(cx, cy, c='steelblue', s=70, zorder=4)
    for idx, (x, y) in enumerate(cust_xy, start=1):
        ax.text(x+0.3, y+0.3, str(idx), fontsize=8)

    colors = plt.cm.tab10.colors
    for ridx, route in enumerate(routes, start=1):
        pts = np.array([depot] + [cust_xy[c-1] for c in route] + [depot])
        ax.plot(pts[:,0], pts[:,1], linewidth=2.2,
                color=colors[ridx % len(colors)], label=f'R{ridx}: {route}')

    ax.set_title(f"{inst['instance_id']} -- {title}\ndist={total_d:.4f}", fontsize=13)
    ax.legend(fontsize=8, loc='best')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

for iid, r in results.items():
    inst_plot = next(i for i in instances if i['instance_id'] == iid)
    plot_routes(inst_plot, r['routes_final'], title='Hybrid Solver')

## Solution Output (Challenge Submission Format)

In [ ]:
for iid, r in results.items():
    print(format_routes_text(r['routes_final'], instance_id=iid))
    print()

## Resource Usage Table

Required submission format for qubit and gate counts.

In [ ]:
print(f"{'Instance':12s} | {'Qubits':>8s} | {'Gates':>8s} | {'Elapsed':>12s} | Method")
print('-' * 65)
for iid, r in results.items():
    meta_list = r.get('all_meta_qaoa', [])
    max_q = max((m.get('n_qubits', 0) for m in meta_list if isinstance(m, dict)), default=0)
    total_g = sum(m.get('n_gates', 0) for m in meta_list if isinstance(m, dict))
    elapsed = sum(m.get('elapsed_s', 0) for m in meta_list if isinstance(m, dict))
    methods = {m.get('method', '?') for m in meta_list if isinstance(m, dict)}
    print(f"{iid:12s} | {max_q:>8d} | {total_g:>8d} | {elapsed:>11.4f}s | {','.join(methods)}")

## QAOA Demonstration (Explicit Quantum Run)

To test QAOA on larger neighborhoods natively, set qubit budget >= 49 so k=7.
This requires `qiskit_aer` and takes ~30-120s per route on a statevector simulator.

**Uncomment the cell below to run.**

In [ ]:
# Uncomment to run actual QAOA on a 7-customer neighborhood (k=7, 49 qubits)
#
# from quantum_improve import demo_qaoa_neighborhood
#
# inst4 = instances[3]  # Instance 4
# D4 = build_distance_matrix(inst4['nodes'])
# route4 = results['Instance4']['routes_final'][0]
#
# if len(route4) >= 7:
#     neigh = route4[0:7]
#     la = 0  # depot as left anchor
#     ra = route4[7] if len(route4) > 7 else 0
#     diag = demo_qaoa_neighborhood(neigh, D4, la, ra, reps=2, shots=1024, restarts=1)
#     print('QAOA neighborhood result:')
#     for k, v in diag.items():
#         print(f'  {k}: {v}')
# else:
#     print('Route has fewer than 7 customers; use larger synthetic instance for QAOA demo.')

print('QAOA demo cell ready -- uncomment to run.')